<a href="https://colab.research.google.com/github/mennamldev/Git-Class/blob/eduibrmen001_TASK-1/ul_coursework2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# if you want to adjust the current working directory to find the downloaded
# CSV file more easily, then the following two lines can be helpful.
import os
os.chdir("..")  # changes the current working directory to the parent directory

In [3]:
# Importing all the necessary libraries
!pip install nltk

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation

Before running the code cell below, download `documents.csv` from the task description in the Moodle course. Then adapt the path in the code cell below (`"path/to/the/file/documents.csv"`) such that it points to the downloaded CSV file.

In [8]:

#First thing is to ensure nltk stopwords are downloaded
nltk.download('stopwords') #Downloads stopwords dataset from NLTK
stop_wor = set(stopwords.words('english')) #Importing stopwords module

#Loading datasets
documents = list(pd.read_csv("/content/documents.csv", header=None).iloc[:, 0]) #'header=None' argument ensures no header is assumed in the CSV

#Text Preprocessing
def readableClean(texT):
    texT = texT.lower()  #lowercase converted
    texT = re.sub(r'\d+', '', texT)  #numbers eliminated
    texT = re.sub(r'[^a-zA-Z\s]', '', texT)  #punctuation eliminated
    words = texT.split()  #list of words from splitted text
    words = [word for word in words if word not in stop_wor]  #Keeps only words with important meaning

    return ' '.join(words) #Join the words back to form clean sentence
#Applying preprocessing function to each document
processed_docs = [readableClean(doc) for doc in documents]

#Converting text to numerical representation
#By using TF-IDF for NMF
Tf_Idf_VectoRIZER = TfidfVectorizer(max_df=0.95, min_df=2, stop_words='english')
#max_df=0.95 means items appearing with more than 95% of the documents are ignored.
#min_df=2 means items that appear with fewer than 2 documents are ignored.
X_tf_idf = Tf_Idf_VectoRIZER.fit_transform(processed_docs) #Fit and transform the preprocessed documents into TF-IDF matrix


#Using Count Vectorizer for LDA
Counting_VectoRizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')# simple representation where each word will be counted.
X_count = Counting_VectoRizer.fit_transform(processed_docs) # Fit and transform the documents to a count-based matrix.

#Apply NMF model
numTOPICS = 5  #Number of topics
nmfMODEL= NMF(n_components=numTOPICS, random_state=42) #Initializing NMF with the specified number of topics
nmfTOPICS = nmfMODEL.fit_transform(X_tf_idf) #Fit the model into TF-IDF matrix and transform the data


#Apply LDA model
LdaMODEL = LatentDirichletAllocation(n_components=numTOPICS, random_state=42) # Initializing LDA with the specified number of topics
LdaTOPICS = LdaMODEL.fit_transform(X_count) #Fit the LDA model into the count-based matrix and transform the data


#Function to display topics
#This function displays the top words of each topic in the model.
def display_topics(model, feature_names, num_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic {topic_idx}: ", " ".join([feature_names[i] for i in topic.argsort()[:-num_top_words - 1:-1]]))

#Display NMF Topics
print("NMF Topics:")
display_topics(nmfMODEL, Tf_Idf_VectoRIZER.get_feature_names_out(), 10) #Displays top 10 words of each NMF topic


#Display LDA Topics
print("\nLDA Topics:")
display_topics(LdaMODEL, Counting_VectoRizer.get_feature_names_out(), 10) #Displays top 10 words of each LDA topic

#Assign documents to topics
#In both NMF and LDA, each document is assigned to the topic with the highest probability.
nmf_DOCS_in_toPICS = np.argmax(nmfTOPICS, axis=1) #For NMF, find the index of the topic with the highest score for each document
lda_DOCS_in_toPICS = np.argmax(LdaTOPICS, axis=1) #For LDA, find the index of the topic with the highest score for each document


#Print sample document-topic assignments
#Print out the topic assignments for the first 10 documents
for i in range(10):
    print(f"Document {i} -> NMF Topic: {nmf_DOCS_in_toPICS[i]}, LDA Topic: {lda_DOCS_in_toPICS[i]}")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


NMF Topics:
Topic 0:  film best awards festival band award films actor actress aviator
Topic 1:  mr labour election said blair party brown government howard tax
Topic 2:  mobile people users phone said technology software music digital tv
Topic 3:  game win england said cup ireland players play team world
Topic 4:  said bn growth economy bank sales year oil economic market

LDA Topics:
Topic 0:  said people mr new software government security users use law
Topic 1:  said music people mobile tv new games technology phone digital
Topic 2:  said world best win game years time new england good
Topic 3:  film said best years mr director actor awards home actress
Topic 4:  said mr labour bn year government election new party people
Document 0 -> NMF Topic: 4, LDA Topic: 4
Document 1 -> NMF Topic: 2, LDA Topic: 0
Document 2 -> NMF Topic: 4, LDA Topic: 4
Document 3 -> NMF Topic: 3, LDA Topic: 2
Document 4 -> NMF Topic: 3, LDA Topic: 0
Document 5 -> NMF Topic: 3, LDA Topic: 2
Document 6 -> NMF 

# Course Work 2

## 1. Which topics can you identify in the dataset?

In [9]:
#Four primary themes arise from the NMF & LDA outcomes: politics, technology, sports, films, and the economy. While the subsequent theme is about politics and elections, the first issue is about films along with film festivals. Sports and international competitions are the subject of the next theme, and business and financial trends are the subject of the fourth.

## 2. Compare the results of the different approaches!

In [10]:
#Although LDA displays conflicting themes, NMF produces themes that are both more apparent and more distinct. Whereas LDA is adaptable but less distinct since it permits words to belong to several themes, NMF offers more structured topics.

## 3. Are you also able to assign documents to topics?

In [11]:
#As illustrated by examples such as document 0 → NMF Topic: 4, the code's document-topic allocation capabilities assigns each document to the relevant subject, enabling examination of which approach best matches the dataset.

# Collaboration Questions

<ol>
    <li>
        <ol type="a">
            <li>
                Did you receive any help whatsoever from anyone in solving this assignment? (yes / no)
            </li>
            <li>
                If yes, give full details (e.g. "Jane Doe explained to me what is asked in Question 1")
            </li>
            <li>Any received help must also be cited in the code with comments indicating the start and the end of code that was not written alone.</li>
        </ol>
    </li>
</ol>

No

<ol start="2">
    <li>
        <ol type="a">
            <li>
                Did you give any help whatsoever to anyone in solving this assignment? (yes / no)
            </li>
            <li>
                If yes, give full details (e.g. "I pointed Joe Smith to slide 10 of the fifth lecture since he didn't know how to proceed with Question 2")
            </li>
        </ol>
    </li>
</ol>

NO